# Computing the energy of water on IBM

In [2]:
from typing import Optional

from joblib import Parallel, delayed
from tqdm import tqdm

import numpy as np
import pickle

import cirq
import openfermion as of

import quimb as qu
import quimb.tensor as qtn
import torch

import qiskit
from qiskit import qasm3
from qiskit_aer import AerSimulator
import qiskit_ibm_runtime
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [3]:
service = qiskit_ibm_runtime.QiskitRuntimeService(name="NERSC-US")

computer = qiskit_ibm_runtime.fake_provider.FakeFez()
sampler = Sampler(computer)

## Read in the Hamiltonian

In [4]:
basis = "sto-3g"  # "6-311g"  # "6-31g"

In [5]:
num_qubits = {
    "sto-3g": 14,
    "6-31g": 26,
    "6-311g": 38,
}
hamiltonian_basis = {
    "sto-3g": "37mHa/sto-3g/sto-3g_hamiltonian_001_adaptiterations.pkl",
    "6-31g": "6-31g/6-31g_hamiltonian_007_adaptiterations.pkl",
    "6-311g": "6-311g/6-311g_hamiltonian_054_adaptiterations.pkl"
}

circuit_basis = {
    "sto-3g": "37mHa/sto-3g/sto-3g_001_adaptiterations.qasm",
    "6-31g": "6-31g/6-31g_hamiltonian_007_adaptiterations.pkl",
    "6-311g": "6-311g/6-311g_054_adaptiterations.qasm"
}


In [6]:
qubit_op = pickle.load(
    open(f"{hamiltonian_basis[basis]}", "rb")
)

In [7]:
# QubitOp from fcidump.
# import pyscf.tools
# from pyscf import ao2mo
# fcidump = pyscf.tools.fcidump.read(f"{basis}/{basis}.fcidump")

# n_orbitals = fcidump.get("NORB")
# num_electrons = fcidump.get("NELEC")
# ecore = fcidump.get("ECORE")
# h1 = fcidump.get("H1")
# h2 = fcidump.get("H2")
# h2 = ao2mo.restore(1, h2, n_orbitals)

# one_body_tensor, two_body_tensor = of.ops.representations.get_tensors_from_integrals(
#     h1, h2
# )

# qubit_op = of.jordan_wigner(
#         of.get_fermion_operator(
#             of.InteractionOperator(
#             constant=ecore,
#             one_body_tensor=one_body_tensor,
#             two_body_tensor=two_body_tensor,
#         )
#     )
# )

In [8]:
def convert_qubitop_to_mpo(qubit_op, n_qubits, max_bond=None, cutoff=1e-10, verbose: bool = False):
    mpo_list = []
    
    for i, (term, coeff) in enumerate(qubit_op.terms.items()):
        ops = [np.eye(2) for _ in range(n_qubits)]
        for idx, pauli_type in term:
            ops[idx] = qu.spin_operator(pauli_type)
        
        term_mpo = qtn.MPO_product_operator(ops) * coeff
        mpo_list.append(term_mpo)

    mpo = mpo_list[0]
    nterms = len(mpo_list)
    for i in range(1, len(mpo_list)):
        if verbose:
            print(f"\rOn term {i + 1} / {nterms}", end="")
        mpo += mpo_list[i]
        mpo.compress(max_bond=max_bond, cutoff=cutoff)
    
    return mpo

In [9]:
# # Parallel version.
# def add_pair(a, b, max_bond, cutoff):
#     a = a + b
#     a.compress(max_bond=max_bond, cutoff=cutoff)
#     return a


# def tree_sum(mpos, max_bond=None, cutoff=1e-10, n_jobs=-1):

#     total_adds = len(mpos) - 1
#     pbar = tqdm(total=total_adds)

#     while len(mpos) > 1:

#         pairs = [(mpos[i], mpos[i+1]) for i in range(0, len(mpos)-1, 2)]

#         results = Parallel(n_jobs=n_jobs, prefer="threads")(
#             delayed(add_pair)(a, b, max_bond, cutoff)
#             for a, b in pairs
#         )

#         pbar.update(len(pairs))

#         if len(mpos) % 2 == 1:
#             results.append(mpos[-1])

#         mpos = results

#     pbar.close()
#     return mpos[0]


# def build_term(term, coeff, n_qubits):
#     term_dict = dict(term)

#     ops = [
#         qu.spin_operator(term_dict[i]) if i in term_dict else np.eye(2)
#         for i in range(n_qubits)
#     ]

#     return qtn.MPO_product_operator(ops) * coeff


# def convert_qubitop_to_mpo(qubit_op, n_qubits, max_bond=None, cutoff=1e-10):
#     mpo_list = Parallel(n_jobs=-1, prefer="threads")(
#         delayed(build_term)(term, coeff, n_qubits)
#         for term, coeff in qubit_op.terms.items()
#     )
#     return tree_sum(mpo_list, max_bond, cutoff)


# mpo = convert_qubitop_to_mpo(qubit_op, n_qubits=num_qubits[basis], max_bond=16)

In [10]:
# mpo = qu.load_from_disk('mpo_6-311g_16')

In [11]:
max_mpo_bond: int = 64

In [12]:
mpo = convert_qubitop_to_mpo(qubit_op, n_qubits=num_qubits[basis], max_bond=max_mpo_bond, verbose=True)

On term 1086 / 1086

In [13]:
qu.save_to_disk(mpo, f"mpo_{basis}_{max_mpo_bond}")

['mpo_sto-3g_64']

In [14]:
mpo

MatrixProductOperator(tensors=14, indices=41, L=14, max_bond=34)

In [15]:
dmrg = qtn.DMRG2(mpo, bond_dims=[10, 20, 50, 100], cutoffs=1e-10)
dmrg.solve(max_sweeps=10, verbosity=1, tol=1e-8)
ground_state_energy = dmrg.energy
ground_state_mps = dmrg.state  # This is the optimized Matrix Product State (MPS)

print(f"Ground state energy: {ground_state_energy}")

1, R, max_bond=(10/10), cutoff:1e-10


100%|##########################################| 13/13 [00:00<00:00, 121.76it/s]

Energy: (-62.01456687144987-6.394884621840902e-14j) ... not converged.
2, R, max_bond=(10/20), cutoff:1e-10



100%|###########################################| 13/13 [00:00<00:00, 50.25it/s]

Energy: (-62.0151584564323+0j) ... not converged.
3, R, max_bond=(12/50), cutoff:1e-10


...

Energy: (-62.094334600120504+2.4868995751603507e-13j) ... not converged.
4, R, max_bond=(16/100), cutoff:1e-10



100%|##########################################| 13/13 [00:00<00:00, 211.10it/s]

Energy: (-62.09434101610998+1.794120407794253e-13j) ... not converged.
5, R, max_bond=(5/100), cutoff:1e-10



100%|##########################################| 13/13 [00:00<00:00, 326.81it/s]

Energy: (-62.094341016705584+1.6608936448392342e-13j) ... converged!
Ground state energy: (-62.094341016705584+1.6608936448392342e-13j)


## Read in the circuit

In [16]:
base = qasm3.load(f"{circuit_basis[basis]}")
base = qiskit.transpiler.passes.RemoveBarriers()(base)
base.draw(fold=-1, idle_wires=False)

┌───┐                                                                                                                                                                                                                                                                                                                                                                                                                                                               
 0: ┤ X ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
    ├───┤                                                                                                                                                                                                                                                                                                                                                                                                                                                               
 1: ┤ X ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
    ├───┤                                                                                                                                                                                                                                                                                                                                                                                                                                                               
 2: ┤ X ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
    ├───┤                                                                                                                                                                                                                                                                                                                                                                                                                                                               
 3: ┤ X ├───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
    ├───┤┌───┐               ┌───┐┌──────────┐                                                                                                                                                                                                                                                                                                              

## Qiskit MPS simulator with hardware noise model

In [17]:
# sim = AerSimulator.from_backend(computer, method="matrix_product_state")

In [18]:
# to_run = base.copy()
# to_run.measure_all()
# to_run = [to_run]
# to_run = qiskit.transpile(
#     to_run,
#     computer,
# )
# to_run[0].save_matrix_product_state(label="mps")
# to_run[0].draw(fold=-1, idle_wires=False)

In [19]:
# result = sim.run(to_run[0], shots=10_000)

In [20]:
# counts = result.result().get_counts()

In [21]:
# counts

In [22]:
# mps_tensors, mps_singular_values = result.result().data()["mps"]

## Cirq noise model / Quimb MPS

In [23]:
def qasm_to_cirq(qc: qiskit.QuantumCircuit) -> cirq.Circuit:
    # Create Cirq qubits
    qubits = [cirq.LineQubit(i) for i in range(qc.num_qubits)][::-1]

    circuit = cirq.Circuit(cirq.I.on(q) for q in qubits)

    for instruction, qargs, _ in qc.data:
        name = instruction.name
        params = instruction.params
        q = [qubits[qc.qubits.index(qubit)] for qubit in qargs]

        if name == "h":
            circuit.append(cirq.H(q[0]))
        
        elif name == "s":
            circuit.append(cirq.S(q[0]))

        elif name == "x":
            circuit.append(cirq.X(q[0]))

        elif name == "y":
            circuit.append(cirq.Y(q[0]))

        elif name == "z":
            circuit.append(cirq.Z(q[0]))

        elif name == "cx":
            circuit.append(cirq.CNOT(q[0], q[1]))

        elif name == "cz":
            circuit.append(cirq.CZ(q[0], q[1]))

        elif name == "swap":
            circuit.append(cirq.SWAP(q[0], q[1]))

        elif name == "rx":
            circuit.append(cirq.rx(params[0])(q[0]))

        elif name == "ry":
            circuit.append(cirq.ry(params[0])(q[0]))

        elif name == "rz":
            circuit.append(cirq.rz(params[0])(q[0]))

        elif name == "sx":
                circuit.append(cirq.XPowGate(exponent=0.5)(q[0]))

        elif name == "barrier":
            continue
            # ignore barriers

        else:
            raise NotImplementedError(f"Gate not supported: {name}")

    return circuit

In [24]:
circuit = qasm_to_cirq(base)
circuit

/tmp/ipykernel_133592/1257008696.py:7: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for instruction, qargs, _ in qc.data:


0: ────I───X───S───H───────────@───X───────────Rz(-0.5π)─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                               │   │
1: ────I───X───S───H───@───X───X───@───────────@───────────X───────────Rz(-0.5π)─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                       │   │                   │           │
2: ────I───X───S───────X───@───@───X───────────X───────────@───────────H───────────Rz(-0.5π)───────────────────────────────────────────@───Rz(-0.858π)───X^0.5───────────────X───Rz(0.5π)───────────────────────X───Rz(-0.5π)───X^0.5───Rz(-π)──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────X────────────@───────X───────────────X───────────────
                               │   │                                                                                                   │                                     │                                  │                                                                                                                                                                                           │            │       │               │
3: ────I───X───S───────────────X───@───────────H───────────Rz(-0.5π)───────────────────────────────────────@───────────X───────────@───X───H─────────────@───────X───@───X───@───X^0.5──────Rz(-0.5π)───X^0.5───@───Rz(-0.5π)───X^0.5───Rz(-0.642π)───X^0.5───Rz(0.987π)───X───Rz(0.5π)───────────────────────X───Rz(0.5π)───X^0.5────────Rz(-π)────────────────────────────────────────────────X───H───────@────────────X───────@───────────X───@───────────────
                                                                                                           │           │           │                     │       │   │                                                                                                     │                                  │                                                                                 │                                            │
4: ────I───X───S───────────────────────────────────────────X───────────@───────────H───────────Rz(-0.5π)───X───────────@───────────X───@─────────────────X───────@───X───@───X───@──────────Rz(-π)──────X──────────────────────────────────────────────────────────────────@───X^0.5──────Rz(-0.5π)───X^0.5───@───X^0.5──────Rz(0.513π)───X^0.5────@───X^0.5──────Rz(1.01π)───X^0.5────Rz(-π)───@───X^0.5───Rz(0.987π)───X^0.5───Rz(-0.5π)───@───@───────────────
                                                           │           │                                                               │                                 │   │   │                                                                                                                                                                 │                                                                                             │
5: ────I───X───S───────────────────X───────────@───────────@───────────X───────────X───────────@───────────H───────────Rz(-0.5π)───────X───H─────────────────────────────X───@───X─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────X───Rz(0.5π)

In [25]:
def simulate(
    circuit: cirq.Circuit,
    verbose: bool = False,
    seed: Optional[int] = None,
    backend: str = "cpu",
    max_bond: Optional[int] = None,
    cutoff: float = 1e-10,
    ) -> qtn.MatrixProductState:
    rng = np.random.RandomState(seed)

    qubits_to_indices = {q: i for i, q in enumerate(sorted(circuit.all_qubits()))}
    nqubits = len(qubits_to_indices)

    mps = qtn.MPS_computational_state("0" * nqubits, dtype="float64", cyclic=False)

    if backend == "gpu":
        for tensor in mps.tensors:
            tensor.modify(
                apply=lambda x: torch.tensor(x, dtype=torch.complex64, device="cuda")
            )

    num_ops = len(list(circuit.all_operations()))
    for i, op in enumerate(circuit.all_operations()):
        qubit_indices = [qubits_to_indices[q] for q in op.qubits]
        if cirq.has_unitary(op):
            to_apply = qu.qarray(cirq.unitary(op))
        elif cirq.has_mixture(op):
            ps = []
            ops = []
            for (p, o) in cirq.mixture(op):
                ps.append(p)
                ops.append(o)
            op = ops[rng.choice(range(len(ops)), p=ps)]
            to_apply = qu.qarray(op)
        else:
            raise ValueError(f"Cannot apply operation {op}")
        
        if backend == "gpu":
            to_apply = torch.tensor(to_apply, dtype=torch.complex64, device="cuda")

        mps.gate_(
            to_apply,
            qubit_indices,
            contract="swap+split",
            max_bond=max_bond,
            cutoff=cutoff,
        )
        if verbose:
            print(f"\rOp {i + 1} / {num_ops}", end="")

    return mps

In [26]:
max_mps_bond: int = 128

In [27]:
mps = simulate(circuit, max_bond=max_mps_bond, backend="cpu")
mps

MatrixProductState(tensors=14, indices=27, L=14, max_bond=2)

In [28]:
# If on GPU:
# for tensor in mpo.tensors:
#     tensor.modify(
#         apply=lambda x: torch.tensor(x, dtype=torch.complex64, device="cuda")
#     )

In [29]:
def quimb_overlap(mps: qtn.MatrixProductState, mpo: qtn.MatrixProductOperator) -> float:
    mpsH = mps.H
    mps.align_(mpo, mpsH)
    return (mpsH & mpo & mps) ^ ...

In [30]:
exact_energy = quimb_overlap(mps, mpo)
exact_energy

-61.74157812934165

In [31]:
noise_rates = [1e-8, 1e-6, 1e-4, 1e-2]
ntrajectories = 10_000

noisy_energies = []
errors = []
for noise_rate in noise_rates:
    print("Status: On noise rate", noise_rate)
    noisy = circuit.with_noise(cirq.depolarize(noise_rate))
    # print(noisy)

    all_mps = Parallel(n_jobs=-1)(
        delayed(simulate)(noisy, max_bond=max_mps_bond, backend="cpu")
        for _ in tqdm(range(ntrajectories), desc="Simulating trajectories")
    )
    overlaps = [quimb_overlap(mps, mpo) for mps in all_mps]
    noisy_energy = np.average(overlaps)
    noisy_energies.append(noisy_energy)
    errors.append(np.abs(noisy_energy - exact_energy))
    print("Noisy energy:", noisy_energy)
    print("Exact energy:", exact_energy)
    print("Absolute error:", errors[-1])

Status: On noise rate 1e-08


Simulating trajectories: 100%|██████████| 10000/10000 [03:08<00:00, 53.05it/s]


Noisy energy: -61.74157812934164
Exact energy: -61.74157812934165
Absolute error: 7.105427357601002e-15
Status: On noise rate 1e-06


Simulating trajectories: 100%|██████████| 10000/10000 [02:59<00:00, 55.77it/s]


Noisy energy: -61.74139048617708
Exact energy: -61.74157812934165
Absolute error: 0.00018764316456554297
Status: On noise rate 0.0001


Simulating trajectories: 100%|██████████| 10000/10000 [02:59<00:00, 55.85it/s]


Noisy energy: -61.64110343050614
Exact energy: -61.74157812934165
Absolute error: 0.1004746988355052
Status: On noise rate 0.01


Simulating trajectories: 100%|██████████| 10000/10000 [03:01<00:00, 55.19it/s]


Noisy energy: -54.183887522307096
Exact energy: -61.74157812934165
Absolute error: 7.5576906070345515


In [37]:
noisy

0: ────I───D(0.01)[<virtual>]───X───D(0.01)[<virtual>]───S───D(0.01)[<virtual>]───H───D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───@───D(0.01)[<virtual>]───X───────────D(0.01)[<virtual>]───Rz(-0.5π)───D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────D(0.01)[<virtual>]─────────────────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────D(0.01)[<virtual>]──────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]─────────────────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]────────────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]──────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]──────────────D(0.01)[<virtual>]────────────────D(0.01)[<virtual>]────────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]──────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]────────────D(0.01)[<virtual>]────────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]────────────────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───
                                                                                                                                                             │                        │
1: ────I───D(0.01)[<virtual>]───X───D(0.01)[<virtual>]───S───D(0.01)[<virtual>]───H───D(0.01)[<virtual>]───@───D(0.01)[<virtual>]───X───D(0.01)[<virtual>]───X───D(0.01)[<virtual>]───@───────────D(0.01)[<virtual>]───@───────────D(0.01)[<virtual>]───X───────────D(0.01)[<virtual>]───Rz(-0.5π)───D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────D(0.01)[<virtual>]─────────────────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────D(0.01)[<virtual>]──────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]─────────────────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]────────────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]──────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]──────────────D(0.01)[<virtual>]────────────────D(0.01)[<virtual>]────────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]──────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]────────────D(0.01)[<virtual>]────────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]────────────────D(0.01)[<virtual>]───────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───
                                                                                                           │                        │                                                                                  │                                │
2: ────I───D(0.01)[<virtual>]───X───D(0.01)[<virtual>]───S───D(0.01)[<virtual>]───────D(0.01)[<virtual>]───X───D(0.01)[<virtual>]───@───D(0.01)[<virtual>]───@───D(0.01)[<virtual>]───X───────────D(0.01)[<virtual>]───X───────────D(0.01)[<virtual>]───@───────────D(0.01)[<virtual>]───H───────────D(0.01)[<virtual>]───Rz(-0.5π)───D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtual>]───────────────D(0.01)[<virtua

In [32]:
noisy_energies

[np.float64(-61.74157812934164),
 np.float64(-61.74139048617708),
 np.float64(-61.64110343050614),
 np.float64(-54.183887522307096)]

In [33]:
exact_energy

-61.74157812934165

In [34]:
errors

[np.float64(7.105427357601002e-15),
 np.float64(0.00018764316456554297),
 np.float64(0.1004746988355052),
 np.float64(7.5576906070345515)]

In [35]:
basis

'sto-3g'

In [36]:
np.savetxt(f"{basis}_noiserates_errors.txt", np.array([noise_rates, errors]).T)